# glcuda vs llama.cpp - head-to-head on a Tesla T4

The M2.2 numbers so far are glcuda-internal (kernel GB/s, decode tok/s per
format). This notebook puts glcuda next to **llama.cpp built with CUDA** on
the SAME GPU, SAME model files, SAME session - the only honest comparison.

Covers all three formats glcuda supports natively: **Q8_0** (bandwidth-bound
reference), **Q4_K_M** (the native-Q6_K path), **Q4_0** (the 4.5 bpw speed
path). Both engines report prefill + decode tok/s.

Requirements: Colab/Kaggle **GPU T4** runtime + Internet on. Runtime ~40 min
(the CUDA llama.cpp build is ~8-12 min one-time; three 7B models download).


## 0 - GPU + toolchain

In [ ]:
!nvidia-smi --query-gpu=name,compute_cap,memory.total --format=csv
import os, subprocess
# Kaggle path adapter (harmless on Colab).
if os.path.exists('/kaggle') and not os.path.exists('/content'):
    subprocess.run(['ln', '-sfn', '/kaggle/working', '/content'], check=True)
    print('linked /content -> /kaggle/working')


In [ ]:
# Rust toolchain (glcuda build).
import os
if not os.path.exists(os.path.expanduser('~/.cargo/bin/cargo')):
    !curl -sSf https://sh.rustup.rs | sh -s -- -y >/dev/null 2>&1
os.environ['PATH'] = os.path.expanduser('~/.cargo/bin') + ':' + os.environ['PATH']
!cargo --version


## 1 - Clone glcuda + generate the runner example

In [ ]:
%cd /content
if not os.path.exists('/content/gwenland-ai'):
    !git clone -q https://github.com/JinXSuperSolo/gwenland-ai
%cd /content/gwenland-ai
!git pull --ff-only 2>&1 | tail -1


In [ ]:
# The run_glcuda example driver (same one section 8/14 use). max_new_tokens
# and sampling are pinned here so llama.cpp can be told to match them.
example = r'''
use std::time::Instant;
use glcore::engine_trait::{GlEngine, InferInput};
use glcuda::GlcudaEngine;

fn main() -> Result<(), Box<dyn std::error::Error>> {
    let path = std::env::args().nth(1).expect("usage: run_glcuda <model.gguf> [prompt]");
    let prompt = std::env::args().nth(2)
        .unwrap_or_else(|| "Explain what a GPU is in one sentence.".to_string());
    let mut engine = GlcudaEngine::new();
    engine.init()?;
    let t = Instant::now();
    engine.load_model(&path)?;
    println!("[load] {:.2}s", t.elapsed().as_secs_f64());
    let ids = engine.encode_chat(&prompt)?;
    println!("[prompt] {} tokens", ids.len());
    let input = InferInput {
        token_ids: ids, max_new_tokens: 128, temperature: 0.7,
        top_k: 40, top_p: 0.95, repeat_penalty: 1.1,
    };
    let out = engine.stream(input, &|_id, _piece| {})?;
    println!("GLCUDA prefill {:.1} tok/s | decode {:.1} tok/s",
        out.prompt_tokens as f64 / (out.prefill_ms/1e3).max(1e-9),
        out.tokens_generated as f64 / (out.generation_ms/1e3).max(1e-9));
    Ok(())
}
'''
os.makedirs('/content/gwenland-ai/glcuda/examples', exist_ok=True)
open('/content/gwenland-ai/glcuda/examples/run_glcuda.rs', 'w').write(example)
# Ensure the example can find glcore (mirrors section 8b).
import re
mani = '/content/gwenland-ai/glcuda/Cargo.toml'
txt = open(mani).read()
if '[[example]]' not in txt:
    open(mani, 'a').write('\n[[example]]\nname = "run_glcuda"\npath = "examples/run_glcuda.rs"\n')
!cargo build --release -p glcuda --example run_glcuda 2>&1 | tail -2


## 2 - Build llama.cpp **with CUDA** (fast path)

Official llama.cpp has no Linux-CUDA binary (Windows-CUDA + Linux-CPU only),
so we build - but trimmed hard so it's ~3-5 min, not 15:

- **nvcc via pip wheel** (`nvidia-cuda-nvcc-cu12`, ~30s) instead of the apt
  toolkit (~2-3 min) - only if Colab's image lacks nvcc.
- **one target** (`llama-bench` + `llama-quantize`), not the whole suite.
- **one arch as real code** (`75-real` = T4 SASS only, no PTX) - nvcc runs
  once per file, no extra virtual-arch pass.
- **ccache** so a re-run after a session reset is near-instant.


In [ ]:
# --- 2a: get nvcc (fast) + detect the GPU arch ---
import os, subprocess, glob, shutil

def find_nvcc():
    n = shutil.which('nvcc')
    if n: return n
    hits = sorted(glob.glob('/usr/local/cuda*/bin/nvcc')
                  + glob.glob('/usr/local/lib/python*/dist-packages/nvidia/cuda_nvcc/bin/nvcc')
                  + glob.glob('/usr/local/lib/python*/site-packages/nvidia/cuda_nvcc/bin/nvcc'))
    if hits:
        os.environ['PATH'] = os.path.dirname(hits[-1]) + ':' + os.environ['PATH']
        return hits[-1]
    return None

nvcc = find_nvcc()
if not nvcc:
    # Colab ships the CUDA runtime but often not nvcc. The pip wheel is far
    # faster than the apt toolkit and matches Colab's CUDA 12 runtime.
    print('nvcc not found - installing via pip wheel (fast)...')
    !pip install -q nvidia-cuda-nvcc-cu12 nvidia-cuda-runtime-cu12 2>&1 | tail -1
    nvcc = find_nvcc()
    if not nvcc:  # last resort: apt
        print('pip wheel miss - falling back to apt toolkit...')
        !apt-get -qq install -y cuda-toolkit-12-2 2>&1 | tail -1
        nvcc = find_nvcc()
print('nvcc:', nvcc or 'NOT FOUND')
if nvcc:
    os.environ['CUDACXX'] = nvcc
    print(subprocess.run([nvcc,'--version'],capture_output=True,text=True).stdout.strip().splitlines()[-1])

cc = subprocess.run(['nvidia-smi','--query-gpu=compute_cap','--format=csv,noheader'],
                    capture_output=True, text=True).stdout.strip().splitlines()[0]
os.environ['LLAMA_ARCH'] = cc.replace('.', '')   # e.g. 7.5 -> 75 (T4)
print('GPU compute cap:', cc, '-> CUDA arch', os.environ['LLAMA_ARCH'])
# ccache makes a rebuild after a session reset near-instant.
!which ccache >/dev/null 2>&1 || apt-get -qq install -y ccache 2>&1 | tail -1


In [ ]:
# --- 2b: configure + build (trimmed), real error on failure ---
%cd /content
if not os.path.exists('/content/llama.cpp'):
    !git clone --depth 1 -q https://github.com/ggml-org/llama.cpp

ARCH = os.environ.get('LLAMA_ARCH', '75')
# "<arch>-real" = build SASS for exactly this GPU, skip the PTX/virtual pass
# (roughly halves nvcc work). ccache launcher for fast re-runs.
cfg = subprocess.run(
    f'cmake -B /content/llama.cpp/build /content/llama.cpp '
    f'-DCMAKE_BUILD_TYPE=Release -DGGML_CUDA=ON '
    f'-DCMAKE_CUDA_ARCHITECTURES={ARCH}-real '
    f'-DGGML_NATIVE=ON -DLLAMA_CURL=OFF '
    f'-DCMAKE_CUDA_COMPILER_LAUNCHER=ccache -DCMAKE_CXX_COMPILER_LAUNCHER=ccache',
    shell=True, capture_output=True, text=True)
if cfg.returncode != 0:
    print('=== CONFIGURE FAILED ===')
    print(cfg.stdout[-3000:]); print(cfg.stderr[-3000:])
else:
    print('configure ok - building llama-bench + llama-quantize...')
    import time; t0 = time.time()
    bld = subprocess.run(
        'cmake --build /content/llama.cpp/build --target llama-bench llama-quantize -j$(nproc)',
        shell=True, capture_output=True, text=True)
    if bld.returncode != 0:
        print('=== BUILD FAILED - real error below ===')
        for line in (bld.stdout + bld.stderr).splitlines():
            if any(k in line.lower() for k in ('error','nvcc fatal','undefined','no such','unsupported')):
                print(line)
        print('--- stderr tail ---'); print(bld.stderr[-1500:])
    else:
        print(f'BUILD OK in {time.time()-t0:.0f}s')
        !ls -lh /content/llama.cpp/build/bin/llama-bench /content/llama.cpp/build/bin/llama-quantize


## 3 - Download the models (Q8_0, Q4_K_M) + convert a pure Q4_0

In [ ]:
%cd /content/gwenland-ai
REPO = 'https://huggingface.co/bartowski/Qwen2.5-7B-Instruct-GGUF/resolve/main'
def fetch(url, out):
    if not os.path.exists(out):
        rc = os.system(f'wget -q -O {out} {url}')
        if rc != 0:
            print('DOWNLOAD FAILED:', out); return
    print(subprocess.run(['ls','-lh',out], capture_output=True, text=True).stdout.strip())

fetch(f'{REPO}/Qwen2.5-7B-Instruct-Q8_0.gguf',  'q8.gguf')
fetch(f'{REPO}/Qwen2.5-7B-Instruct-Q4_K_M.gguf', 'q4km.gguf')
# Pure Q4_0: public "Q4_0" files are mixed quants (Q4_1 ffn_down = glcore
# Unknown(3) load error), so requantize the Q8_0 to a single-type Q4_0.
if not os.path.exists('q4_0.gguf'):
    !/content/llama.cpp/build/bin/llama-quantize --allow-requantize --pure q8.gguf q4_0.gguf Q4_0 2>&1 | tail -2
!ls -lh q4_0.gguf


## 4 - The comparison

For each format: glcuda's own timing (its `run_glcuda`) vs `llama-bench`
with `-n 128` (matches glcuda's max_new_tokens) and `-p 512` prefill. Both
pin all layers to GPU. llama.cpp decode = the `tg128` row, prefill = `pp512`.

Note the numbers are not bit-identical workloads (glcuda uses a chat prompt,
llama-bench uses synthetic tokens) but both measure the same thing - steady
-state decode tok/s and prefill throughput on the T4 - which is the honest
comparison. Decode is what matters most; it's the bandwidth-bound steady
state both engines converge to.


In [ ]:
import re
LB = '/content/llama.cpp/build/bin/llama-bench'
PROMPT = "Explain how quantum computing differs from classical computing, covering qubits, superposition, entanglement, and key algorithms."

def glcuda(model):
    r = subprocess.run(['target/release/examples/run_glcuda', model, PROMPT],
                       capture_output=True, text=True)
    m = re.search(r'GLCUDA prefill ([\d.]+) tok/s \| decode ([\d.]+)', r.stdout)
    if not m:
        print('  glcuda parse miss for', model, '- stderr tail:', r.stderr.strip()[-200:])
    return (float(m.group(1)), float(m.group(2))) if m else (None, None)

def llama(model):
    # -ngl 99 = all layers on GPU. llama-bench markdown table has a "test"
    # column (pp512 / tg128) and a final "t/s" column like "123.45 +/- 0.67".
    # Grab the first float on whichever row names the test - robust to column
    # count and spacing across llama.cpp versions.
    r = subprocess.run([LB, '-m', model, '-ngl', '99', '-p', '512', '-n', '128', '-r', '3'],
                       capture_output=True, text=True)
    pp = tg = None
    for line in r.stdout.splitlines():
        if '|' not in line:
            continue
        cells = [c.strip() for c in line.strip().strip('|').split('|')]
        test = next((c for c in cells if c in ('pp512', 'tg128')), None)
        if not test:
            continue
        ts = re.search(r'([\d.]+)', cells[-1])  # last col = "t/s +/- err"
        if ts:
            if test == 'pp512': pp = float(ts.group(1))
            else:               tg = float(ts.group(1))
    if pp is None and tg is None:
        print('  llama parse miss for', model, '- stdout tail:', r.stdout.strip()[-300:])
    return pp, tg

rows = []
for fmt, model in [('Q8_0','q8.gguf'), ('Q4_K_M','q4km.gguf'), ('Q4_0','q4_0.gguf')]:
    if not os.path.exists(model):
        print(fmt, '(missing)'); continue
    print('benchmarking', fmt, '...')
    gp, gd = glcuda(model)
    lp, ld = llama(model)
    rows.append((fmt, gp, gd, lp, ld))

print()
print(f'{"format":<8} {"engine":<10} {"prefill t/s":>12} {"decode t/s":>11}  {"decode ratio":>12}')
print('-'*58)
for fmt, gp, gd, lp, ld in rows:
    ratio = (gd/ld) if (gd and ld) else None
    print(f'{fmt:<8} {"glcuda":<10} {gp or 0:>12.1f} {gd or 0:>11.1f}  {(f"{ratio:.2f}x" if ratio else "-"):>12}')
    print(f'{fmt:<8} {"llama.cpp":<10} {lp or 0:>12.1f} {ld or 0:>11.1f}')
    print('-'*58)


## What to read

- **Decode** is the headline. On Q8_0 both are bandwidth-bound and should
  nearly tie (~29 t/s on a T4). The interesting rows are Q4_K_M and Q4_0:
  how close glcuda's native SoA kernels get to llama.cpp's mature CUDA
  kernels on the same bytes.
- **Prefill**: glcuda uses the tensor-core MMA GEMM (M2.1 Task B); llama.cpp
  has years of kernel tuning. A gap here is expected and is the next lever.
- If a cell prints 0.0, its parse missed - re-run `llama-bench -m <model>
  -ngl 99` by hand and read the tg128/pp512 rows directly.
